# SignLab — Explorer les classifieurs de panneaux
Ce notebook montre comment auditer les données, entraîner plusieurs modèles sur les mêmes partitions et lire leurs erreurs. Le parcours commence par une démonstration synthétique. Place tes images dans `data/images/` et passe `DEMO_SYNTHETIQUE` à `False` pour utiliser le dataset réel.

Les deux modes ont des dossiers de sortie différents. Les résultats synthétiques servent à vérifier le parcours technique.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import sys
import tempfile
import numpy as np

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "configs/default.json").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from signlab.data import group_split, load_dataset
from signlab.experiment import benchmark, finalize, load_config
from signlab.registry import CATALOG, build_model
from signlab.features import Preprocessor
from signlab.artifacts import load_bundle, predict_image
from PIL import Image

DEMO_SYNTHETIQUE = True
TOUS_LES_MODELES = False
RAPIDE = True
config = load_config(ROOT / "configs/default.json")
if DEMO_SYNTHETIQUE:
    from scripts.synthetic_demo import create_fixture
    scratch = Path(tempfile.mkdtemp(prefix="signlab-notebook-"))
    create_fixture(scratch / "images", per_class=18)
    config.update(data_dir=str(scratch / "images"), data_kind="synthetic",
                  near_duplicate_distance=-1, image_size=8, pca_components=8)
    run = scratch / "run"
else:
    run = ROOT / "runs" / "notebook-reel"
print("Type de données :", config["data_kind"])
print("Sortie :", run)

## Inventaire et partitions
Un groupe représente des images liées. Les proportions peuvent s’écarter des fractions cibles quand des groupes sont indivisibles. Inspecte les effectifs avant de lancer une comparaison.

In [ ]:
dataset = load_dataset(config)
splits = group_split(dataset, config["validation_fraction"], config["test_fraction"], config["split_seed"])
for name, indices in splits.items():
    print(name, dict(Counter(dataset.records[i]["class"] for i in indices)))
print("Incidents de chargement :", len(dataset.issues))
print("Empreinte :", dataset.fingerprint)
print("Origines déclarées :", Counter(row["source"] for row in dataset.records))

## Comparer sur validation
La référence majoritaire permet de mesurer la contribution du signal appris. Le macro-F1 est la moyenne des F1 calculés classe par classe. Active `TOUS_LES_MODELES` pour le catalogue complet. Désactive `RAPIDE` pour parcourir plusieurs réglages et graines. Le test reste réservé à l’étape finale.

In [ ]:
model_names = None if TOUS_LES_MODELES else ["majority", "logistic", "linear_svm", "knn", "decision_tree", "mlp"]
rows = benchmark(config, run, models=model_names, quick=RAPIDE, progress=lambda name: print(name))
for row in rows:
    print(f"{row['model']:20}  F1={row['validation_macro_f1_mean']:.3f}  "
          f"temps={row['fit_seconds_mean']:.2f}s  taille={row['artifact_bytes']:,} octets")

In [ ]:
selection = json.loads((run / "selection.json").read_text(encoding="utf-8"))
trials = json.loads((run / "trials.json").read_text(encoding="utf-8"))
chosen = rows[0]
trial = next(t for t in trials if t["model"] == chosen["model"]
             and t["params"] == chosen["params"] and t["seed"] == chosen["seed"])
print("Modèle choisi :", selection["model"])
print("Matrice de validation (lignes réelles, colonnes prédites) :")
print(np.array(trial["validation"]["confusion_matrix"]))
for metrics in trial["validation"]["per_class"]:
    print(metrics)
print("Écart macro-F1 train – validation :", round(chosen["train_macro_f1_mean"] - chosen["validation_macro_f1_mean"], 3))
try:
    from IPython.display import SVG, display
except ImportError:
    print("Figure :", run / "figures" / f"{chosen['model']}_validation.svg")
else:
    display(SVG(filename=str(run / "figures" / f"{chosen['model']}_validation.svg")))

## Expliquer les erreurs et la capacité
Une confusion récurrente peut signaler un problème de cadrage, de couleur, de représentation ou d’étiquette. Ouvre quelques images de validation mal classées et note les motifs observables. Le prochain exemple isole une limite de capacité connue : les deux classes de XOR ne sont pas séparables par une droite.

In [ ]:
artifact = load_bundle(run / chosen["artifact"])
model, preprocessor, _ = artifact
idx = splits["validation"]
predicted = model.predict(preprocessor.transform(dataset.X[idx]))
errors = [(dataset.records[i]["path"], dataset.class_names[dataset.y[i]], dataset.class_names[p])
          for i, p in zip(idx, predicted) if p != dataset.y[i]]
print("Erreurs de validation (fichier, vraie classe, prédiction) :", errors[:8])
with Image.open(Path(config["data_dir"]) / dataset.records[idx[0]]["path"]) as photo:
    output = predict_image(artifact, photo)
print("Exemple d’inférence sur la validation :", output["class"], output["score_notice"])

X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])
for name in ("logistic", "kernel_svm_rbf"):
    candidate = build_model(name, {"C": 10, "gamma": 2} if name.startswith("kernel") else None)
    candidate.fit(X_xor, y_xor)
    print(name, candidate.predict(X_xor), "accuracy d’entraînement :", candidate.score(X_xor, y_xor))

## Test final
Passe la variable ci-dessous à `True` après avoir figé la représentation, les candidats et les choix de validation. Pour le dataset réel, documente l’origine et le nombre de groupes avant d’interpréter le score. Le test peut être exécuté une fois par dossier d’expérience.

In [ ]:
FINALISER = False
if FINALISER:
    results = finalize(run)
    for result in results:
        print(result["model"], result["metrics"])
else:
    print("Test réservé. Active FINALISER après avoir figé les choix.")